# Case Study 2 — Transformer-Based Topic Modeling with BERTopic

## Business problem
We have many unlabeled documents and want to discover:

> **What are people mainly talking about?**

This notebook uses **BERTopic** with a cleaner preprocessing flow so topic keywords are not dominated by stopwords.

### Revised flow

```text
Raw documents
   ↓
Light text cleaning
   ↓
Transformer embeddings
   ↓
Semantic clustering
   ↓
Stopword-aware topic vectorizer
   ↓
Cleaner topic words / phrases
   ↓
Human interpretation
```

The important design choice is:

> Keep enough natural language for good Transformer embeddings, but remove common stopwords when BERTopic creates the topic keywords.

## 1. Install libraries

In [1]:
# Install/update the libraries needed for this notebook.
# - BERTopic: topic modeling
# - sentence-transformers: Transformer-based document embeddings
# - scikit-learn: dataset utilities and CountVectorizer
# -q keeps installation output shorter; -U updates packages if needed.
%pip install -q -U bertopic sentence-transformers scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 104.8 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


## 2. Import libraries

In [2]:
# Import the libraries used across the BERTopic workflow.
# We use:
# - re for text cleaning with regular expressions
# - NumPy/Pandas for arrays and tables
# - Counter for simple word-frequency inspection
# - scikit-learn for the dataset and topic-word vectorizer
# - SentenceTransformer to convert documents into semantic vectors
# - BERTopic to discover and describe topics.
import re
import numpy as np
import pandas as pd

# Counter will help us inspect the most frequent raw words later.
from collections import Counter
# Load a public text dataset and a CountVectorizer for topic-word extraction.
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer

# Transformer embedding model used by BERTopic.
from sentence_transformers import SentenceTransformer
# Main BERTopic class.
from bertopic import BERTopic

# 3. Load a public dataset

We use four categories from **20 Newsgroups**:

- Space
- Baseball
- Computer Graphics
- Medicine

We remove headers, footers and quoted replies because they often contain repeated words such as `writes`, `article`, `subject`, etc. that can create artificial topics.

In [3]:
# Select four 20 Newsgroups categories so the discovered topics
# have reasonably clear themes for a classroom demonstration.
categories = [
    "sci.space",
    "rec.sport.baseball",
    "comp.graphics",
    "sci.med"
]

# Download/load only the training portion of the selected categories.
# remove=(...) strips headers, footers and quoted replies so the model
# focuses more on the actual article content.
news = fetch_20newsgroups(
    subset="train",
    categories=categories,
    remove=("headers", "footers", "quotes"),
    random_state=42
)

raw_documents = news.data
# Extract only the raw article text from the downloaded dataset.

print("Raw documents:", len(raw_documents))
# Check how many documents were loaded and inspect one raw example.
print("\nExample:")
print(raw_documents[0][:700])

Raw documents: 2368

Example:
The O's just lost to the Rangers a few minutes ago I was not too happy about
the pitching of Rick Sutcliffe (6 runs in 6 innings, 5 in the 3?) This puts
me in remembering the 1990 O's season. After '89 we didn't do much over
winter and we wound up in 5th. Now I know that Mussina, McDonald, and Rhodes
are better pitching prospects than Ballard and Milacki but are any other
Oriole fans scared out there?

Admiral Steve C. Liu


# 4. Light text cleaning

We remove obvious noise:

- URLs
- email addresses
- HTML-like tags
- special characters
- extra spaces

We **do not aggressively remove stopwords here**.

### Why?

The embedding model uses sentence context.

For example:

```text
"This medicine is not effective"
```

Removing too much before embedding can weaken the meaning.

So we use:

```text
Light cleaning before embeddings
+
Stopword removal later for topic keywords
```

In [4]:
# Define a light-cleaning function.
# Important BERTopic idea:
# we remove obvious noise but keep natural sentence context because
# the Transformer embedding model benefits from seeing the full wording.
def clean_text(text):
# Remove web URLs.
    text = re.sub(r"http\S+|www\.\S+", " ", text)
# Remove email-like strings.
    text = re.sub(r"\S+@\S+", " ", text)
# Remove simple HTML-like tags.
    text = re.sub(r"<.*?>", " ", text)
# Remove most special characters while keeping letters, numbers,
# apostrophes and whitespace.
    text = re.sub(r"[^A-Za-z0-9'\s]", " ", text)
# Replace repeated whitespace with one space and trim the ends.
    text = re.sub(r"\s+", " ", text).strip()
# Return the cleaned document.
    return text


# Apply the cleaning function to every raw document.
documents = [
    clean_text(doc)
    for doc in raw_documents
]

# Remove almost-empty documents
documents = [
    doc
    for doc in documents
# Keep only documents with at least 10 words so almost-empty
# documents do not create weak/noisy topics.
    if len(doc.split()) >= 10
]

# Smaller classroom sample
documents = documents[:1500]

print("Documents used:", len(documents))
# Keep a smaller sample so the classroom demonstration runs faster.
print("\\nCleaned example:")
print(documents[0][:700])

Documents used: 1500
\nCleaned example:
The O's just lost to the Rangers a few minutes ago I was not too happy about the pitching of Rick Sutcliffe 6 runs in 6 innings 5 in the 3 This puts me in remembering the 1990 O's season After '89 we didn't do much over winter and we wound up in 5th Now I know that Mussina McDonald and Rhodes are better pitching prospects than Ballard and Milacki but are any other Oriole fans scared out there Admiral Steve C Liu


# 5. Create a stopword-aware topic vectorizer

This is the key correction.

BERTopic uses this vectorizer when deciding which words best describe each topic.

```python
CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=3
)
```

### `stop_words="english"`
Removes common words such as:

```text
the, and, is, of, to, for
```

### `ngram_range=(1, 2)`
Allows:

- single words: `space`
- two-word phrases: `space shuttle`

This often makes topics easier to understand.

### `min_df=3`
A word/phrase must appear in at least 3 documents.

Lower value:
- keeps more rare words
- can add noise

Higher value:
- cleaner vocabulary
- may remove useful niche words

In [5]:
# Create the CountVectorizer used by BERTopic to describe each topic.
# The Transformer embeddings are used for semantic clustering;
# this vectorizer is mainly used later to produce readable topic words.
vectorizer_model = CountVectorizer(
# Remove common English stopwords such as 'the', 'is', 'and'.
    stop_words="english",
# Allow both single words and two-word phrases.
    ngram_range=(1, 2),
# Keep a term even if it appears in only one aggregated topic document.
# min_df=1 is safe here because BERTopic may create only a few topics.
    min_df=1
)

# 6. Load the Transformer embedding model

`all-MiniLM-L6-v2` converts each complete document into a semantic vector.

Documents with similar meanings should get similar vectors.

In [6]:
# Load the pretrained SentenceTransformer embedding model.
# It converts each complete document into a dense semantic vector.
embedding_model = SentenceTransformer(
# all-MiniLM-L6-v2 is a compact sentence/document embedding model
# that works well for a classroom BERTopic example.
    "all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# 7. Create BERTopic

### `min_topic_size=20`

A topic should contain roughly at least 20 documents.

Lower:
- more / smaller topics
- more noise possible

Higher:
- fewer / broader topics

The important addition is:

```python
vectorizer_model=vectorizer_model
```

This tells BERTopic to use our cleaned vocabulary when generating topic keywords.

In [7]:
# Create the BERTopic model by combining:
# 1. Transformer embeddings for semantic meaning
# 2. the stopword-aware CountVectorizer for readable topic words.
topic_model = BERTopic(
# Use the SentenceTransformer model loaded above.
    embedding_model=embedding_model,
# Use our vectorizer when extracting representative words/phrases.
    vectorizer_model=vectorizer_model,
# Require roughly 20 documents before forming a normal topic.
# Smaller groups may be merged or treated as outliers.
    min_topic_size=20,
# Show progress messages while BERTopic runs.
    verbose=True
)

# 8. Fit the model

Conceptually:

```text
Documents
   ↓
Transformer embeddings
   ↓
Group documents with similar meaning
   ↓
Use stopword-cleaned vocabulary
   ↓
Find representative topic words
```

In [8]:
# Fit BERTopic on the cleaned documents and assign a topic to each document.
# topics = topic ID assigned to each document.
# probabilities = model confidence/probability information when available.
topics, probabilities = topic_model.fit_transform(
    documents
)

# Inspect the first few topic assignments.
# Topic -1, if present, represents outlier documents with no strong topic.
print("First 20 topic assignments:")
print(topics[:20])

2026-09-26 09:38:27,014 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/47 [00:00<?, ?it/s]

2026-09-26 09:38:29,734 - BERTopic - Embedding - Completed ✓
2026-09-26 09:38:29,735 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-26 09:38:42,676 - BERTopic - Dimensionality - Completed ✓
2026-09-26 09:38:42,677 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-26 09:38:42,720 - BERTopic - Cluster - Completed ✓
2026-09-26 09:38:42,726 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-26 09:38:43,380 - BERTopic - Representation - Completed ✓


First 20 topic assignments:
[1, 2, 2, 2, 1, 2, 0, 1, 0, 2, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1]


# 9. View discovered topics

`Topic = -1` means the document did not fit strongly into any discovered topic.

That is normal in BERTopic.

In [9]:
# Ask BERTopic for a summary table of all discovered topics.
# Typical columns include Topic, Count, Name and representative terms.
topic_info = topic_model.get_topic_info()

# Display the first 12 topic rows for inspection.
display(topic_info.head(12))

,Topic,Count,Name,Representation,Representative_Docs
0,0,778,0_space_data_image_graphics,"[space, data, image, graphics, nasa, program, ...",[Archive name space data Last modified Date 93...
1,1,367,1_year_00_game_team,"[year, 00, game, team, good, games, think, bet...",[I thought I'd post my predicted standings sin...
2,2,355,2_health_don_food_medical,"[health, don, food, medical, use, people, pati...",[cut here HICNet Medical Newsletter Page 13 Vo...


# 10. Inspect one topic

The topic words should now look much more meaningful.

Example:

```text
space, nasa, orbit, shuttle, launch
```

instead of:

```text
the, of, and, to, is
```

In [10]:
# Choose one discovered topic to inspect in detail.
topic_id = 0

# get_topic(topic_id) returns representative words/phrases
# together with their BERTopic relevance scores.
print(
    topic_model.get_topic(topic_id)
)

[('space', np.float64(0.02176708172631212)), ('data', np.float64(0.011351401015310177)), ('image', np.float64(0.010944121490981727)), ('graphics', np.float64(0.009961096248229218)), ('nasa', np.float64(0.008903316025710277)), ('program', np.float64(0.00862623809122563)), ('available', np.float64(0.008060139846291274)), ('like', np.float64(0.007757031971100553)), ('information', np.float64(0.007651595259045167)), ('software', np.float64(0.007616895322124965))]


# 11. Create a clean topic summary

We show the top 8 words/phrases for each topic.

In [11]:
# Build a compact summary table for the first few discovered topics.
# Each row will contain the topic ID, number of documents
# and its top representative words/phrases.
rows = []

# Loop through the first topic IDs listed by BERTopic.
for topic_id in topic_info["Topic"].head(10):
# Skip topic -1 because -1 represents outliers/no strong topic.
    if topic_id == -1:
        continue

# Get the representative words and their scores for this topic.
    topic_words = topic_model.get_topic(topic_id)

# Keep only the first 8 representative words/phrases.
    top_words = [
        word
        for word, score in topic_words[:8]
    ]

# Look up how many documents BERTopic assigned to this topic.
    count = int(
        topic_info.loc[
            topic_info["Topic"] == topic_id,
            "Count"
        ].iloc[0]
    )

    rows.append({
# Store one clean summary row for this topic.
        "topic_id": topic_id,
        "document_count": count,
        "top_words": ", ".join(top_words)
    })

topics_df = pd.DataFrame(rows)

# Convert the collected rows into a DataFrame and display it.
display(topics_df)

,topic_id,document_count,top_words
0,0,778,"space, data, image, graphics, nasa, program, a..."
1,1,367,"year, 00, game, team, good, games, think, better"
2,2,355,"health, don, food, medical, use, people, patie..."


# 12. Inspect real documents from a topic

Top words are only a clue.

Reading actual documents helps confirm whether the topic is genuinely coherent.

In [12]:
# Choose a topic and inspect real documents assigned to it.
# This is important because topic words alone do not tell the full story.
selected_topic = 0

# Find document positions whose assigned topic equals selected_topic.
# [:3] keeps only the first three matching documents for easy inspection.
example_indices = [
    i
    for i, topic in enumerate(topics)
    if topic == selected_topic
][:3]

# Print a short preview of each selected document.
for i in example_indices:
    print("=" * 80)
    print(documents[i][:700])
    print()

I am currently looking for a 3D graphics library that runs on MS Windows 3 1 Are there any such libraries out there other than Visuallib It must run on VGA and should not require any other add on graphics cards For Visuallib will it run with Metaware High C compiler v3 0 Any email contact for the author of Visuallib Any help would be much appreciated Thanks

Try graPHIGS from IBM It is an excellent package Doug

Sorry not so the changes in sunrise and sunset times are not quite synchronized For example neither the earliest sunrise nor the latest sunset comes on the longest day of the year



# 13. Add human-readable topic names

BERTopic discovers groups.

Humans interpret and name them.

Example:

```text
nasa, orbit, spacecraft, launch
        ↓
Space / NASA
```

The mapping below is only an example. Update it after looking at the actual topic output.

In [13]:
# Give selected discovered topics human-readable names.
# These labels are interpretation aids, not labels learned automatically.
# Update this mapping after inspecting the actual topic words/documents.
topic_names = {
    0: "Space / NASA",
    1: "Medicine / Health",
    2: "Baseball / Sports",
    3: "Computer Graphics"
}

# Map BERTopic's numeric topic IDs to our human-readable names.
topic_info["human_name"] = (
    topic_info["Topic"].map(topic_names)
)

# Display BERTopic's original topic information beside the human names.
display(
    topic_info[
        ["Topic", "Count", "human_name", "Name"]
    ].head(10)
)

,Topic,Count,human_name,Name
0,0,778,Space / NASA,0_space_data_image_graphics
1,1,367,Medicine / Health,1_year_00_game_team
2,2,355,Baseball / Sports,2_health_don_food_medical


# 14. Test new documents

A new document can be assigned to one of the discovered topics.

In [14]:
# Create a few new example documents that were not used in fitting.
# They represent the broad themes we expect the model to have discovered.
new_documents = [
    "NASA announced a spacecraft mission that will study Mars.",
    "The doctor recommended a new treatment for the patient.",
    "The baseball team won after scoring three runs.",
    "The graphics card supports advanced 3D rendering."
]

# Apply exactly the same light-cleaning function to the new text.
clean_new_documents = [
    clean_text(text)
    for text in new_documents
]

# Use the fitted BERTopic model to assign each new document
# to one of the discovered topics.
new_topics, new_probabilities = (
    topic_model.transform(clean_new_documents)
)

# Print each new document and its predicted topic.
for text, topic in zip(
    new_documents,
    new_topics
):
    print(text)
    print("Predicted topic ID:", topic)

# If the model assigned a normal topic, show its top words.
# If the topic is -1, treat the document as an outlier/no strong match.
    if topic != -1:
        print(
            "Topic words:",
            topic_model.get_topic(topic)[:5]
        )
    else:
        print("Outlier / no strong topic")

    print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-09-26 09:38:43,796 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-09-26 09:38:46,883 - BERTopic - Dimensionality - Completed ✓
2026-09-26 09:38:46,884 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-09-26 09:38:46,886 - BERTopic - Cluster - Completed ✓


NASA announced a spacecraft mission that will study Mars.
Predicted topic ID: 0
Topic words: [('space', np.float64(0.02176708172631212)), ('data', np.float64(0.011351401015310177)), ('image', np.float64(0.010944121490981727)), ('graphics', np.float64(0.009961096248229218)), ('nasa', np.float64(0.008903316025710277))]

The doctor recommended a new treatment for the patient.
Predicted topic ID: 2
Topic words: [('health', np.float64(0.014309536935303449)), ('don', np.float64(0.01156118044340409)), ('food', np.float64(0.011481216068665579)), ('medical', np.float64(0.011431599034777262)), ('use', np.float64(0.011419585961542111))]

The baseball team won after scoring three runs.
Predicted topic ID: 1
Topic words: [('year', np.float64(0.022946463078497462)), ('00', np.float64(0.01977590106955874)), ('game', np.float64(0.018375781347809306)), ('team', np.float64(0.017461265265570763)), ('good', np.float64(0.014920590044413888))]

The graphics card supports advanced 3D rendering.
Predicted top

# 15. See why stopword removal matters

This cell compares:

1. common words in the lightly cleaned documents
2. common terms after the stopword-aware vectorizer

This is useful for seeing exactly what changed.

In [15]:
# Compare raw frequent words with the stopword-aware vocabulary.
# This demonstrates why stopword removal is useful for topic labels.
raw_words = []

# Collect every raw word from the lightly cleaned documents.
for doc in documents:
    raw_words.extend(
        doc.lower().split()
    )

# Show the 20 most common raw words before vectorizer stopword removal.
print("Most common raw words:")
print(
    Counter(raw_words).most_common(20)
)

# Fit the same CountVectorizer to the documents.
# The vectorizer removes English stopwords and can create 1-2 word phrases.
X = vectorizer_model.fit_transform(documents)

# Get the vocabulary terms created by CountVectorizer.
terms = vectorizer_model.get_feature_names_out()
# Sum each vocabulary column to obtain its total frequency.
counts = np.asarray(X.sum(axis=0)).ravel()

# Find the indices of the 20 highest-frequency terms.
top_idx = counts.argsort()[-20:][::-1]

# Display the most common terms after the vectorizer's stopword handling.
print("\nMost common words/phrases after stopword removal:")
print(
    [
        (terms[i], int(counts[i]))
        for i in top_idx
    ]
)

Most common raw words:
[('the', 13280), ('of', 6703), ('to', 6573), ('and', 6121), ('a', 6107), ('in', 4380), ('is', 3867), ('i', 3234), ('for', 3084), ('that', 2863), ('it', 2521), ('on', 2009), ('0', 1850), ('this', 1796), ('be', 1774), ('you', 1697), ('are', 1686), ('with', 1647), ('have', 1602), ('as', 1421)]

Most common words/phrases after stopword removal:
[('space', 861), ('like', 514), ('don', 471), ('just', 432), ('time', 428), ('data', 424), ('use', 419), ('know', 405), ('year', 395), ('think', 394), ('good', 378), ('image', 375), ('new', 329), ('graphics', 325), ('people', 320), ('does', 312), ('information', 311), ('program', 308), ('10', 299), ('years', 298)]


# 16. Final takeaway

The improved BERTopic flow is:

```text
Raw text
   ↓
Remove obvious noise
   ↓
Keep natural sentence context
   ↓
Transformer embeddings
   ↓
Semantic clustering
   ↓
Remove stopwords for topic-word extraction
   ↓
Cleaner topic keywords
   ↓
Inspect documents
   ↓
Human topic names
```

## Why this is better than removing all stopwords at the start

Transformer embeddings benefit from natural sentence context.

BERTopic's topic labels benefit from a cleaner vocabulary.

So the practical pattern is:

> **Light cleaning for embeddings + stronger cleaning for topic-word representation.**

In [16]:
# LDA uses a different representation from BERTopic.
# BERTopic keeps natural text for Transformer embeddings;
# LDA works with tokenized Bag-of-Words, so we preprocess the text here.
#1. LDA-specific preprocessing
# Import Gensim's basic tokenizer.
from gensim.utils import simple_preprocess
# Import Gensim's built-in stopword set.
from gensim.parsing.preprocessing import STOPWORDS


def preprocess_for_lda(text):
# Define the preprocessing applied specifically for LDA.

    # Convert text into lowercase word tokens
    tokens = simple_preprocess(
        text,
# Convert text to lowercase alphabetic tokens, remove accents
# and ignore very short tokens.
        deacc=True,
        min_len=3
    )

    # Remove stopwords
    tokens = [
        word
# Remove common stopwords before building the LDA vocabulary.
        for word in tokens
        if word not in STOPWORDS
    ]

    return tokens


processed_docs = [
    preprocess_for_lda(doc)
# Apply LDA preprocessing to the same documents used for BERTopic.
    for doc in documents
]


print("Original article:")
print(documents[0][:500])
# Compare one original document with the tokens LDA will actually use.

print("\nProcessed tokens:")
print(processed_docs[0][:50])

Original article:
The O's just lost to the Rangers a few minutes ago I was not too happy about the pitching of Rick Sutcliffe 6 runs in 6 innings 5 in the 3 This puts me in remembering the 1990 O's season After '89 we didn't do much over winter and we wound up in 5th Now I know that Mussina McDonald and Rhodes are better pitching prospects than Ballard and Milacki but are any other Oriole fans scared out there Admiral Steve C Liu

Processed tokens:
['lost', 'rangers', 'minutes', 'ago', 'happy', 'pitching', 'rick', 'sutcliffe', 'runs', 'innings', 'puts', 'remembering', 'season', 'winter', 'wound', 'know', 'mussina', 'mcdonald', 'rhodes', 'better', 'pitching', 'prospects', 'ballard', 'milacki', 'oriole', 'fans', 'scared', 'admiral', 'steve', 'liu']


In [17]:
# Create the mapping between words and integer IDs required by Gensim LDA.
#2. Create the Gensim dictionary
# Import Gensim's Dictionary class.
from gensim.corpora import Dictionary


dictionary = Dictionary(
# Build the dictionary from all preprocessed documents.
    processed_docs
)


print(
# Check the initial vocabulary size before frequency filtering.
    "Vocabulary before filtering:",
    len(dictionary)
)

Vocabulary before filtering: 20326


In [18]:
# Filter the LDA vocabulary to remove terms that are too rare
# or so common that they are unlikely to distinguish topics.
#Remove very rare and very common words
# no_below=3: word must appear in at least 3 documents.
# no_above=0.5: remove words appearing in more than 50% of documents.
# keep_n=20000: cap the retained vocabulary at 20,000 terms.
dictionary.filter_extremes(
    no_below=3,
    no_above=0.5,
    keep_n=20000
)


# Inspect how much the vocabulary was reduced.
print(
    "Vocabulary after filtering:",
    len(dictionary)
)

Vocabulary after filtering: 6587


In [19]:
# Convert every tokenized document into LDA's Bag-of-Words format.
# doc2bow represents a document as pairs:
# (word_id, count). Word order is not retained.
#4. Convert articles into Bag-of-Words
corpus = [
    dictionary.doc2bow(doc)
    for doc in processed_docs
]

# Inspect both representations for the same first document.

print("First document tokens:")
print(processed_docs[0][:20])
# The Bag-of-Words output contains integer word IDs and frequencies.

print("\nBag-of-Words:")
print(corpus[0][:20])

First document tokens:
['lost', 'rangers', 'minutes', 'ago', 'happy', 'pitching', 'rick', 'sutcliffe', 'runs', 'innings', 'puts', 'remembering', 'season', 'winter', 'wound', 'know', 'mussina', 'mcdonald', 'rhodes', 'better']

Bag-of-Words:
[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1), (12, 2), (13, 1), (14, 1), (15, 1), (16, 1), (17, 1), (18, 1), (19, 1)]


In [20]:
# Train the LDA topic model on the Bag-of-Words corpus.
#5. Train the LDA model
# Import Gensim's LdaModel.
from gensim.models import LdaModel


lda_model = LdaModel(
# corpus supplies the document word-count representation.
    corpus=corpus,
# id2word lets Gensim translate word IDs back to readable words.
    id2word=dictionary,

# Ask LDA to discover 3 latent topics.
    num_topics=3,

# Fix the random seed so the classroom run is more reproducible.
    random_state=42,

# passes=8 means LDA makes 8 passes over the complete corpus.
# iterations=100 controls optimization work within each pass/update.
    passes=8,
    iterations=100,

# Let Gensim learn the document-topic prior automatically.
    alpha="auto",

# Also retain per-word topic assignment information.
    per_word_topics=True
)

In [21]:
# Create a helper function that converts LDA's topic output
# into an easy-to-read Pandas table.
# Display discovered topics
def display_topics(
# model = trained LDA model; num_words = words to show per topic.
    model,
    num_words=10
):

    rows = []
# Collect one summary row for every discovered topic.

    for topic_id, words in model.show_topics(
# show_topics returns topic IDs plus (word, weight) pairs.
# formatted=False keeps the output structured rather than a text string.
        num_topics=-1,
        num_words=num_words,
        formatted=False
    ):

        rows.append({
# Keep the topic ID and join only the readable word names.
            "topic_id": topic_id,

            "top_words": ", ".join([
                word
                for word, weight in words
            ])
        })

    return pd.DataFrame(rows)
# Return the completed topic summary table.


lda_topics_df = display_topics(
# Generate the table for our fitted LDA model.
    lda_model,
    num_words=10
)

display(lda_topics_df)
# Display the discovered LDA topics.

,topic_id,top_words
0,0,"image, graphics, use, data, software, availabl..."
1,1,"think, year, like, good, people, time, know, b..."
2,2,"space, nasa, launch, earth, satellite, shuttle..."
